In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score
import torch
from datasets import Dataset
import evaluate
import pandas as pd
import gc
import numpy as np
import json
import nltk
from utils.config import CV_DATA
import random
import re
from utils.model_pipelines import extract_skills_from_text

🚀 When to Use BERT Instead of T5?

✔ When answers are short and present in the context (factual QA).
✔ When speed is important (BERT is faster than T5).
✔ When you don’t want to generate new text (T5 can hallucinate).

If your task is summarization or generative QA, T5 is better.
If you want extractive, fact-based QA, go with BERT, RoBERTa, or DeBERTa.

Want me to help set up fine-tuning for a BERT model? 🚀

In [2]:
dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'

#qa model
qa_type_model_name= "google/flan-t5-small"
qa_type_model_result= '.temp/model_results/google-flan-t5-small'
qa_type_model= '.temp/model/google-flan-t5-small'

MAX_TOKEN_SIZE= 512

In [ ]:
CV_DATA.keys()

### Preprocessing

In [ ]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type", "answer"]]
df["answer"]= df["answer"].fillna("")
df.info()

In [5]:
# df= df[df["question_type"]== "skills"]

In [6]:
def map_context(row):
    if row['question_type'] == 'skills':
        text= row['question']
        context= []
        skill_list= extract_skills_from_text(text)
        if not skill_list:
            return CV_DATA["skills"]
        context= [line for line in CV_DATA["skills"].split("\n") if any(re.search(rf'\b{skill}\b', line, re.IGNORECASE) for skill in skill_list)]
        return '\n'.join(context)
    return CV_DATA.get(row['question_type'], None)

In [ ]:
df['context'] = df.apply(map_context, axis=1)
df.sample(5)

In [ ]:
df= df.sample(frac=1).reset_index(drop=True)
df.head()

In [ ]:
df.info()

In [9]:
# df= df.sample(40)

### Retraing Preparations:

In [10]:
# Preprocess data
def preprocess_data(row):
    input_text = f"You are a candidate filling job application form answer the question based on the given information.\nQuestion: {row['question']}\nContext: {row['context']}"
    target_text = row['answer']
    return {"input_text": input_text, "target_text": target_text}

processed_data = df.apply(preprocess_data, axis=1)
dataset = Dataset.from_pandas(pd.DataFrame(processed_data.tolist()))

In [11]:
# Split data into train and test
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(qa_type_model_name)
tokenizer.model_max_length = MAX_TOKEN_SIZE
def tokenize_data(example):
    input_encodings = tokenizer(
        example["input_text"], truncation=True, padding="max_length", max_length=MAX_TOKEN_SIZE
    )
    target_encodings = tokenizer(
        example["target_text"], truncation=True, padding="max_length", max_length=512
    )
    input_encodings["labels"] = [
        [-100 if token == tokenizer.pad_token_id else token for token in labels]
        for labels in target_encodings["input_ids"]
    ]
    return input_encodings



train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

In [13]:
def extend_t5_positional_embeddings(model, new_max_length= MAX_TOKEN_SIZE):
    """ Properly extends T5 model to handle longer sequences """
    model.config.n_positions = new_max_length  # This may be ignored since T5 relies on relative embeddings
    model.config.max_length = new_max_length
    model.resize_token_embeddings(model.config.vocab_size)  # Ensures embedding size matches vocab
    
    return model

# Train

In [14]:
model = T5ForConditionalGeneration.from_pretrained(qa_type_model_name)
# model = extend_t5_positional_embeddings(model, new_max_length= MAX_TOKEN_SIZE)
# model.config.max_position_embeddings = MAX_TOKEN_SIZE

In [15]:
training_args = TrainingArguments(
    output_dir=qa_type_model_result,  
    eval_strategy="epoch",
    save_strategy="epoch",
    save_steps=500,
    learning_rate=2e-4,
    num_train_epochs=20,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    # logging_dir="./logs",
    # logging_strategy="epoch",  # Ensure logs are tied to epoch
    # logging_steps=9999999,  # Large number so logs almost never print
    save_total_limit=3,
    warmup_steps=500,
    weight_decay=0.01,
    adam_epsilon=1e-8,
    max_grad_norm=0.5,
    dataloader_pin_memory=False,
    load_best_model_at_end=True,
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)    

torch.cuda.empty_cache()
gc.collect()

trainer.train()

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(qa_type_model)
tokenizer.save_pretrained(qa_type_model)